# Synthetic Data Validation: Geometric Diagnostics vs Known True k₀

**Goal**: Verify that the 9 empirical findings from the real-data report reproduce
when we control the true factor rank k₀.

**Findings to validate**:
1. Sharpe ↑ in P (saturates ~4096) — KMZ Theorem 1
2. OOS R² ↑ in P (benign overfitting)
3. Spectral gap cliff at k₀ — Theorem 6.1(iii)
4. Low drift anisotropy ρ_θ → high Sharpe — Theorem 6.1(i)
5. All metrics ↑ in z — ridge–curvature chain
6. z=0 never finds optimal k — flat Hessian
7. d_proj ↓ in k at z=0 — Theorem 6.1(i)
8. Moderate d_proj is optimal (Goldilocks zone)
9. High Sharpe with erank collapse at k > k₀

**Strategy**: 3 focused sweeps instead of full Cartesian product:
- **Sweep A** (P=1024, k×z): findings 3,5,6,7,8,9 — ~24 jobs
- **Sweep B** (k∈{12,24}, P×z): findings 1,2 — ~12 jobs
- **Sweep C** (P=1024, z∈{10,50}, all k): finding 4 (anisotropy) — covered by A

In [4]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# Add the project root so "from src.xxx" imports work
PROJECT_ROOT = os.path.dirname(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.rff import RandomFourierFeatures
from src.generate_synthetic_panel import generate_synthetic_panel
from src._grass_worker import (
    build_rff_inputs, build_jobs, run_sweep, aggregate, make_results_df,
    _z_label,
)

print('Imports OK')

Imports OK


## 1. Generate synthetic panel with known true k₀ = 12

In [5]:
TRUE_K0 = 12

df_synth, truth, char_cols = generate_synthetic_panel(
    T   = 360,
    N   = 50,
    m   = 30,
    k0  = TRUE_K0,
    seed = 42,
    signal_eigenvalues = np.array([
        8.0, 6.0, 4.5, 3.5, 3.0, 2.5, 2.0, 1.8, 1.6, 1.4, 1.2, 1.05
    ]),
    sigma_eps      = 1.0,
    factor_ar1     = 0.5,
    factor_vol     = 0.3,
    w_drift_scale  = 0.02,
    z_corr         = 0.3,
    hetero_strength = 0.3,
)

# Shared constants
NUM_FACTORS_LIST = [4, 8, 12, 16, 20, 24, 28, 32]
GAMMA      = 0.25
WINDOW_LEN = 24
NUM_ITER_RFF = 1   # single RFF seed — halves compute

# Dense char matrix for RFF
X_chars = (
    df_synth[char_cols]
    .apply(lambda s: s.fillna(s.mean()), axis=0)
    .astype(np.float64)
)

print(f"Panel shape   : {df_synth.shape}")
print(f"Date range    : {df_synth.index.get_level_values('date').min().date()} -> "
      f"{df_synth.index.get_level_values('date').max().date()}")
print(f"Assets        : {df_synth.index.get_level_values('permno').nunique()}")
print(f"True k0       : {truth['true_k0']}")
print(f"Signal lambdas: {truth['signal_eigenvalues']}")
print(f"X_chars shape : {X_chars.shape}")

Panel shape   : (18000, 32)
Date range    : 1994-01-01 -> 2023-12-01
Assets        : 50
True k0       : 12
Signal lambdas: [8.   6.   4.5  3.5  3.   2.5  2.   1.8  1.6  1.4  1.2  1.05]
X_chars shape : (18000, 30)


---
## 2. Sweep A — Fix P=1024, sweep k x z

**Jobs**: 8 k-values x 3 z-values x 1 seed = **24 jobs**

Validates findings 3, 4, 5, 6, 7, 8, 9

In [ ]:
P_A = [1024]
K_A = NUM_FACTORS_LIST
Z_A = [0, 10, 50]

rff_A = build_rff_inputs(X_chars, df_synth, P_A, NUM_ITER_RFF, GAMMA, RandomFourierFeatures)
jobs_A = build_jobs(rff_A, K_A, P_A, Z_A, NUM_ITER_RFF, WINDOW_LEN)
print(f"Sweep A: {len(jobs_A)} jobs (P=1024, k x z)")

raw_A = run_sweep(jobs_A, verbose=True)
agg_A = aggregate(raw_A)
df_A  = make_results_df(agg_A)

print(f"\nSweep A done. Shape: {df_A.shape}")
df_A[['avg_sharpe', 'avg_spectral_gap', 'avg_erank', 'avg_subspace_stability']].round(4)

Sweep A: 24 jobs (P=1024, k x z)
Dispatching 24 jobs across 7 workers (cost range 8192–185364, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   1 out of  24 | elapsed: 27.1min remaining: 622.4min


In [3]:
agg_A

NameError: name 'agg_A' is not defined

### Sweep A plots: Finding 3 — Spectral gap cliff at k0

In [1]:
fig, ax = plt.subplots(figsize=(8, 5))
for z_val in Z_A:
    z_lab = _z_label(z_val)
    try:
        sub = df_A.xs((1024, z_lab), level=('P', 'z'))
        ax.plot(sub.index, sub['avg_spectral_gap'], 'o-', label=f'z={z_val}')
    except KeyError:
        pass

ax.axvspan(TRUE_K0 - 0.5, TRUE_K0 + 2.5, alpha=0.15, color='orange', label=f'k0 = {TRUE_K0}')
ax.set_xlabel('Factors k')
ax.set_ylabel('Spectral gap lambda_k / lambda_1')
ax.set_title(f'Finding 3: Spectral Gap Cliff (P=1024, true k0={TRUE_K0})')
ax.legend()
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

### Sweep A plots: Finding 5 — All metrics co-move with z

In [ ]:
k_plot = TRUE_K0
P_plot = 1024

metrics_z = []
for z_val in Z_A:
    z_lab = _z_label(z_val)
    try:
        row = df_A.loc[(k_plot, P_plot, z_lab)]
        metrics_z.append({
            'z': z_val,
            'Sharpe': row['avg_sharpe'],
            'd_proj': row['avg_subspace_stability'],
            'theta_max': row['avg_max_principal_angle'],
            'erank': row['avg_erank'],
        })
    except KeyError:
        pass

df_z = pd.DataFrame(metrics_z).set_index('z')
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

ax1.plot(df_z.index, df_z['d_proj'], 'o-b', label='d_proj')
ax1.plot(df_z.index, df_z['theta_max'], 's--r', label='theta_max')
ax2.plot(df_z.index, df_z['Sharpe'], 'D-g', label='Sharpe')

ax1.set_xlabel('Regularisation z')
ax1.set_ylabel('Geometric metrics')
ax2.set_ylabel('Sharpe')
ax1.set_title(f'Finding 5: All Metrics Co-Move with z (k={k_plot}, P={P_plot})')
ax1.legend(loc='upper left'); ax2.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Sweep A plots: Finding 7 — d_proj decays with k at z=0

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
z_lab = _z_label(0)

try:
    sub = df_A.xs((1024, z_lab), level=('P', 'z'))
    ax.plot(sub.index, sub['avg_subspace_stability'], 'o-b', label='d_proj')
    ax.set_xlabel('Factors k')
    ax.set_ylabel('Mean d_proj')
    ax.axvline(TRUE_K0, ls='--', c='orange', label=f'true k0={TRUE_K0}')
    ax.set_title(f'Finding 7: d_proj Decays with k at z=0 (P=1024)')
    ax.legend()
except KeyError:
    print('No z=0 data.')

plt.tight_layout()
plt.show()

### Sweep A plots: Finding 6 — z=0 erratic vs z>0 clean peaks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: z=0 erratic
z_lab0 = _z_label(0)
try:
    sub = df_A.xs((1024, z_lab0), level=('P', 'z'))
    axes[0].plot(sub.index, sub['avg_sharpe'], 'o-r', label='P=1024, z=0')
except KeyError:
    pass
axes[0].set_title('z=0 (no ridge) — erratic')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Sharpe')
axes[0].axvline(TRUE_K0, ls='--', c='orange')
axes[0].legend()

# Right: z>0 clean
for z_val in [10, 50]:
    z_lab = _z_label(z_val)
    try:
        sub = df_A.xs((1024, z_lab), level=('P', 'z'))
        axes[1].plot(sub.index, sub['avg_sharpe'], 'o-', label=f'P=1024, z={z_val}')
    except KeyError:
        pass
axes[1].set_title('z>=10 (with ridge) — clean peaks')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Sharpe')
axes[1].axvline(TRUE_K0, ls='--', c='orange')
axes[1].legend()

plt.suptitle(f'Finding 6: z=0 Never Finds Optimal k (true k0={TRUE_K0})', y=1.02)
plt.tight_layout()
plt.show()

### Sweep A plots: Finding 9 — erank collapse at k > k0

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for z_val in [10, 50]:
    z_lab = _z_label(z_val)
    try:
        sub = df_A.xs((1024, z_lab), level=('P', 'z'))
        axes[0].plot(sub.index, sub['avg_erank'], 'o-', label=f'z={z_val}')
        axes[1].plot(sub.index, sub['avg_sharpe'], 'o-', label=f'z={z_val}')
    except KeyError:
        pass

axes[0].axvline(TRUE_K0, ls='--', c='orange', label=f'true k0={TRUE_K0}')
axes[0].plot(K_A, K_A, 'k--', alpha=0.3, label='erank=k')
axes[0].set_xlabel('k'); axes[0].set_ylabel('erank(Sigma_f)')
axes[0].set_title('erank vs k'); axes[0].legend()

axes[1].axvline(TRUE_K0, ls='--', c='orange', label=f'true k0={TRUE_K0}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Sharpe')
axes[1].set_title('Sharpe vs k'); axes[1].legend()

plt.suptitle(f'Finding 9: erank Collapse at k > k0 (P=1024)', y=1.02)
plt.tight_layout()
plt.show()

### Sweep A plots: Matched-pair comparison z=0 vs z=10

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
z0_lab  = _z_label(0)
z10_lab = _z_label(10)

bar_w = 1.5
for metric, ax_i, ylabel in [
    ('avg_subspace_stability', axes[0], 'Mean d_proj'),
    ('avg_sharpe', axes[1], 'Sharpe'),
]:
    for z_lab, color, label, offset in [
        (z0_lab,  'red',  'z=0',  -0.8),
        (z10_lab, 'blue', 'z=10',  0.8),
    ]:
        try:
            sub = df_A.xs((1024, z_lab), level=('P', 'z'))
            ax_i.bar(sub.index + offset, sub[metric],
                     width=bar_w, color=color, alpha=0.7, label=label)
        except KeyError:
            pass
    ax_i.axvline(TRUE_K0, ls='--', c='orange')
    ax_i.set_xlabel('k'); ax_i.set_ylabel(ylabel); ax_i.legend()

plt.suptitle(f'Matched Pairs (P=1024): Illusory vs Virtuous', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Sweep B — Fix k in {12, 24}, sweep P x z

**Jobs**: 2 k-values x 3 P-values x 2 z-values x 1 seed = **12 jobs**

Validates findings 1, 2 (Sharpe and R2 monotonically increase with P)

In [ ]:
P_B = [256, 1024, 4096]
K_B = [TRUE_K0, 24]         # at k0 and past the cliff
Z_B = [10, 50]              # virtuous only (z=0 is noise here)

rff_B = build_rff_inputs(X_chars, df_synth, P_B, NUM_ITER_RFF, GAMMA, RandomFourierFeatures)
jobs_B = build_jobs(rff_B, K_B, P_B, Z_B, NUM_ITER_RFF, WINDOW_LEN)
print(f"Sweep B: {len(jobs_B)} jobs (k in {K_B}, P x z)")

raw_B = run_sweep(jobs_B, verbose=True)
agg_B = aggregate(raw_B)
df_B  = make_results_df(agg_B)

print(f"\nSweep B done. Shape: {df_B.shape}")
df_B[['avg_sharpe', 'avg_r2_oos']].round(4)

### Sweep B plots: Finding 1 & 2 — Sharpe & R2 increase with P

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for k_val in K_B:
    for z_val in Z_B:
        z_lab = _z_label(z_val)
        try:
            sub = df_B.xs(z_lab, level='z').xs(k_val, level='k')
            axes[0].plot(sub.index, sub['avg_sharpe'], 'o-',
                         label=f'k={k_val}, z={z_val}')
            axes[1].plot(sub.index, sub['avg_r2_oos'], 'o-',
                         label=f'k={k_val}, z={z_val}')
        except KeyError:
            pass

axes[0].set_xlabel('RFF dimension P'); axes[0].set_ylabel('Annualised Sharpe')
axes[0].set_title('Finding 1: Sharpe increases with P'); axes[0].legend()
axes[0].set_xscale('log')

axes[1].set_xlabel('RFF dimension P'); axes[1].set_ylabel('OOS R2')
axes[1].set_title('Finding 2: R2 increases with P (less negative = better)'); axes[1].legend()
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

---
## 4. Combine sweeps & compute anisotropy

Merge Sweep A + B for the cross-cutting plots (findings 4, 8, correlation table).

In [ ]:
# Merge results (drop duplicates if any overlap at P=1024)
df_results = pd.concat([df_A, df_B])
df_results = df_results[~df_results.index.duplicated(keep='first')].sort_index()

# Merge raw aggregations for anisotropy computation
results_agg = {**agg_A, **agg_B}

# Compute anisotropy
aniso_rows = []
for key, val in results_agg.items():
    k, P, z_lab = key
    all_aniso = []
    for pa_series in val['principal_angles_series']:
        for angles in pa_series:
            if len(angles) > 0 and angles[0] > 1e-8:
                all_aniso.append(np.mean(angles) / angles[0])
    mean_aniso = np.mean(all_aniso) if all_aniso else np.nan
    aniso_rows.append({'k': k, 'P': P, 'z': z_lab, 'anisotropy': mean_aniso})

df_aniso = pd.DataFrame(aniso_rows).set_index(['k', 'P', 'z'])
df_results = df_results.join(df_aniso)

print(f"Combined results: {df_results.shape}")
df_results[['avg_sharpe', 'avg_spectral_gap', 'avg_erank', 'anisotropy']].head(15)

### Finding 4: Drift anisotropy vs Sharpe

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Virtuous regime only (z > 0)
mask_virt = ~df_results.index.get_level_values('z').str.contains('0e\\+00')
virt = df_results[mask_virt].dropna(subset=['anisotropy'])
illu = df_results[~mask_virt].dropna(subset=['anisotropy'])

ax.scatter(virt['anisotropy'], virt['avg_sharpe'], c='steelblue',
           label='Virtuous (z>0)', alpha=0.7)
ax.scatter(illu['anisotropy'], illu['avg_sharpe'], c='red', marker='^',
           label='Illusory (z=0)', alpha=0.7)

if len(virt) > 3:
    r = virt[['anisotropy', 'avg_sharpe']].corr().iloc[0, 1]
    ax.set_title(f'Finding 4: Anisotropy vs Sharpe (r_virt = {r:.2f})')
else:
    ax.set_title('Finding 4: Anisotropy vs Sharpe')

ax.set_xlabel('Drift anisotropy rho_theta = mean(theta)/max(theta)')
ax.set_ylabel('Annualised Sharpe')
ax.legend()
plt.tight_layout()
plt.show()

### Finding 8: Goldilocks zone — Sharpe vs d_proj

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sc = ax.scatter(
    df_results['avg_subspace_stability'],
    df_results['avg_sharpe'],
    c=np.log10(df_results.index.get_level_values('P').astype(float)),
    cmap='viridis', alpha=0.7, edgecolors='k', linewidths=0.3,
)
plt.colorbar(sc, label='log10(P)')
ax.set_xlabel('d_proj (subspace stability)')
ax.set_ylabel('Annualised Sharpe')
ax.set_title('Finding 8: Goldilocks Zone')
plt.tight_layout()
plt.show()

---
## 5. Summary correlation table

In [ ]:
cols = ['avg_subspace_stability', 'avg_max_principal_angle',
        'avg_mean_principal_angle', 'avg_erank', 'avg_spectral_gap', 'anisotropy']

mask_virt = ~df_results.index.get_level_values('z').str.contains('0e\\+00')

corr_all  = df_results[cols + ['avg_sharpe']].corr()['avg_sharpe'].drop('avg_sharpe')
corr_virt = df_results.loc[mask_virt, cols + ['avg_sharpe']].corr()['avg_sharpe'].drop('avg_sharpe')

df_corr = pd.DataFrame({'All data': corr_all, 'Virtuous (z>0)': corr_virt})
print("\n=== Metric-Sharpe Correlations ===")
print(df_corr.round(3))

## 6. Full results table for paper Section 8

In [ ]:
top = df_results.nlargest(10, 'avg_sharpe')[[
    'avg_sharpe', 'avg_spectral_gap', 'avg_erank', 'avg_subspace_stability', 'anisotropy'
]]
print("\n=== Top 10 Configurations by Sharpe ===")
print(top.round(4))